# API Quicktest

A small, manual-config alternative to `notebooks/api_demo.ipynb` --
no AWS credentials or `boto3` lookups required. Set `API_URL` (and
`API_KEY`, if the backend requires one) by hand below, then run the
notebook. Points at anything: the deployed EC2 instance, or
`http://localhost:5000/predict` when running `server/app.py` locally
for development (see `docs/deployment.md`'s "Running locally" section).

**Known limitation:** on held-out test data, this model achieves ~49-50%
accuracy -- effectively no better than a coin flip. This notebook is a
pipeline smoke test, not a source of real trading signal.

While this shows that there does not appear to be much signal detected by
our role-based transcript parsing strategy within the the transcripts
themselves, this could potentially be used as a tool for guidance on
language within an earnings call.

The introduction of other metrics alongside the earnings call transcript
parsing could also be another possibility in a forward revision of the
tool, such as prior estimates and actuals of earlier earnings periods.

If an additional featureset was implemented, along with the necessary
API Keys for our transcript and stock data feeds, the ticker itself
for a company which has recently had an earnings call would be one
possibility for an API.  Another would be a set of tickers in a
configuration file, ideally with some form of push-based or polling front
end.

Further, the strategy of trying to utilize an index to standardize
other financial measurements could also hold promise when attempting to
account for unseen variables, as well as for portfolio risk planning
when utilizing such a tool.

## 1. Configure

In [ ]:
# Edit these by hand -- no AWS credentials needed for this notebook.
#
# Find the current deployment's values with:
#   aws cloudformation describe-stacks --stack-name ml26-fintc \
#     --query "Stacks[0].Outputs"
#   aws ssm get-parameter --name /ml26-fintc/api-key --with-decryption \
#     --query Parameter.Value --output text

# API_URL = "http://<instance-ip>:5000/predict"  # edit me
API_KEY = ""  # edit me (blank is fine for local dev -- server/app.py skips the check when unset)

## 2. A minimal example transcript

No file dependencies -- this is the same minimal example verified in
`deploy/frontend/examples.html`. Swap in a real transcript (e.g. via
`pandas.read_parquet("data/generated/labeled_transcripts.parquet")`,
as `api_demo.ipynb` does) if you want a realistic prediction instead of
just confirming the pipeline runs.

In [18]:
sample_transcript = "Jane Doe -- Chief Executive Officer\nThis is a smoke test."

## 3. Call the API

In [19]:
import requests

response = requests.post(
    API_URL,
    headers={"Content-Type": "application/json", "x-api-key": API_KEY},
    json={"transcript": sample_transcript},
    timeout=90,
)

print(f"status: {response.status_code}")
result = response.json()
result

status: 200


{'attention_weights': [1.0],
 'num_turns': 1,
 'predicted_label': 1,
 'probability': 0.6280983090400696}